# The Webster horn: converging a taper onto a known answer

A **tapered duct** is a *composite* element: Nefes discretizes a smooth area profile $A(x)$ into
$N$ uniform ducts joined by $N$ compact, loss-free area changes, one matching element per
segment interface.  Refining $N$ makes the chain approach the continuous horn.

The companion notebook `acoustics/acoustic_refinement.ipynb` shows that the scattering matrix
*settles* as $N$ grows.  This notebook answers the question that leaves open — **settles on
what?** — by comparing against an independent solution of the horn equations, and then shows how
the built-in refinement helpers let you pick $N$ without doing that comparison every time.

For a passage of area $A(x)$ carrying no mean flow, continuity and momentum give the classical
horn (Webster) system

$$
\frac{\mathrm{d}\widehat{p}}{\mathrm{d}x} = -\mathrm{i}\omega\overline{\varrho}\,\widehat{u},
\qquad
\frac{\mathrm{d}\widehat{u}}{\mathrm{d}x} =
    -\frac{\mathrm{i}\omega}{\overline{\varrho}\,\overline{c}^{\,2}}\,\widehat{p}
    - \widehat{u}\,\frac{A'(x)}{A(x)},
$$

in the $e^{\mathrm{i}\omega t}$ convention.  Integrating $\mathbf{Y}' = \mathbf{M}(x)\mathbf{Y}$
from $\mathbf{Y}(0)=\mathbf{I}$ yields the exact $(p,u)$ transfer matrix of the horn.  That is a
genuinely independent reference: different equations, a different discretization, a different
code path, and no closed-form expression taken on trust.

We use an **exponential horn**, $A(x) = A_{\rm in}e^{mx}$, which flares by 4:1.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.integrate import solve_ivp

import nefes
from nefes.assembly.recover import ES_C, ES_RHO
from nefes.elements import auto_refine, grid_refine
from nefes.elements import catalog as cat
from nefes.plotting import use_nefes_theme
from nefes.solver.report import states_table

use_nefes_theme()

CFG = nefes.perfect_gas(287.0, 1.4)
P0, T0 = 101325.0, 300.0
A_IN, A_OUT, L = 0.01, 0.04, 0.5  # 4:1 flare over half a metre
FLARE = np.log(A_OUT / A_IN) / L  # the exponential rate m


def area(x):
    "Exponential horn profile A(x) = A_in exp(m x)."
    return A_IN * np.exp(FLARE * x)


def horn(n):
    "inlet -> tapered_duct(N segments) -> outlet, at equal pressures so the mean flow is at rest."
    return nefes.Network(
        CFG,
        nodes=[
            cat.total_pressure_inlet(P0, T0),
            cat.tapered_duct(area, length=L, n_segments=n, name="horn"),
            cat.pressure_outlet(P0, T0),
        ],
        edges=[(0, 1, A_IN), (1, 2, A_OUT)],
        mdot_ref=0.5,  # quiescent: no realized flow to derive a mass scale from
    )


C0 = np.sqrt(1.4 * 287.0 * T0)
print(f"exponential horn:  A {A_IN} -> {A_OUT} m^2 over {L} m,  flare m = {FLARE:.4f} 1/m")
print(f"horn cut-on frequency  m c / (4 pi) = {FLARE * C0 / (4.0 * np.pi):.1f} Hz")

## The reference: integrate the horn equations

Below the cut-on frequency $mc/4\pi$ an exponential horn does not propagate, so every frequency
we probe is chosen above it.

A free check on the reference itself: for a lossless passage at rest, reciprocity in $(p,u)$
requires $\det\mathbf{T} = A_{\rm in}/A_{\rm out}$ exactly.  If the integration were wrong there
is no reason it would land on that number.

In [ ]:
def webster_tm(freq, rho, c):
    "Exact (p, u) transfer matrix of the horn, by numerically integrating the horn system."
    w = 2.0 * np.pi * freq

    def rhs(_x, y):
        block = np.array(
            [[0.0, -1j * w * rho], [-1j * w / (rho * c * c), -FLARE]],
            dtype=complex,
        )
        return (block @ y.reshape(2, 2)).ravel()

    sol = solve_ivp(rhs, (0.0, L), np.eye(2, dtype=complex).ravel(), rtol=1e-12, atol=1e-14)
    assert sol.success
    return sol.y[:, -1].reshape(2, 2)


FREQS = np.array([120.0, 250.0, 400.0, 650.0])
RHO0 = P0 / (287.0 * T0)

print(f"reciprocity of the reference:  det(T) should be A_in/A_out = {A_IN / A_OUT:.6f}")
for f in FREQS:
    print(f"   f = {f:6.1f} Hz   det(T) = {np.linalg.det(webster_tm(f, RHO0, C0)):.8f}")

## Nefes' transfer matrix across the taper

Nefes reports the two-port in the characteristic amplitudes $(\widehat{f},\widehat{g})$.  With no
mean flow the map to primitives is $\widehat{p} = \overline{\varrho}\,\overline{c}(f+g)$ and
$\widehat{u} = f-g$, and $\overline{\varrho}\,\overline{c}$ is the same at both ends, so a single
similarity transform puts the two matrices in the same variables.

In [ ]:
def nefes_tm(n, freqs):
    "Nefes' (p, u) transfer matrix across the taper at N = n segments."
    sol = horn(n).solve()
    assert sol.converged, (sol.residual_norm, sol.print_residuals())
    est = states_table(sol.problem, sol.x)
    rho, c = float(est[ES_RHO, 0]), float(est[ES_C, 0])
    resp = sol.perturbation_response(np.atleast_1d(np.asarray(freqs, float)), excite=("acoustic",))
    t_char = np.asarray(resp.transfer_matrix(0, 1, basis="char")).reshape(-1, 2, 2)
    basis = np.array([[rho * c, rho * c], [1.0, -1.0]], dtype=complex)  # (p, u) <- (f, g)
    return np.einsum("ij,kjl,lm->kim", basis, t_char, np.linalg.inv(basis)), rho, c


def max_rel_error(n):
    "Largest relative transfer-matrix error against the horn reference, over FREQS."
    t_n, rho, c = nefes_tm(n, FREQS)
    return max(
        np.abs(t_n[j] - webster_tm(f, rho, c)).max() / np.abs(webster_tm(f, rho, c)).max() for j, f in enumerate(FREQS)
    )


t_coarse, _, _ = nefes_tm(4, FREQS)
t_fine, rho, c = nefes_tm(128, FREQS)
print("transfer matrix at 250 Hz\n")
print("N = 4   (coarse)\n", np.round(t_coarse[1], 5), "\n")
print("N = 128 (resolved)\n", np.round(t_fine[1], 5), "\n")
print("Webster reference\n", np.round(webster_tm(250.0, rho, c), 5))

## Convergence

The staircase is *endpoint-biased* — each segment's duct carries its downstream station area,
not the centre value — and the two ends are structurally pinned to the external edge areas.  So
the expected rate is **first order** in $1/N$, not second.  That is what we see: each doubling of
$N$ halves the error, a slope of $-1$ on log--log axes.

In [ ]:
resolutions = [4, 8, 16, 32, 64, 128, 256]
errors = np.array([max_rel_error(n) for n in resolutions])

print(f"{'N':>6}  {'max rel error':>15}  {'observed order':>15}")
for i, n in enumerate(resolutions):
    order = "" if i == 0 else f"{np.log2(errors[i - 1] / errors[i]):.2f}"
    print(f"{n:>6}  {errors[i]:>15.3e}  {order:>15}")

fig = go.Figure()
fig.add_scatter(x=resolutions, y=errors, mode="lines+markers", name="taper vs Webster horn")
fig.add_scatter(
    x=resolutions,
    y=errors[0] * resolutions[0] / np.array(resolutions, dtype=float),
    mode="lines",
    line=dict(dash="dash"),
    name="first order, 1/N",
)
fig.update_xaxes(type="log", title_text="segments N")
fig.update_yaxes(type="log", title_text="max relative error in T")
fig.update_layout(height=460, title="Tapered duct converges on the true horn at first order")
fig.show()

## Picking $N$ without a reference solution

In practice there is no exact answer to compare against — that is the whole point of the solver.
Nefes therefore ships two discretization-convergence helpers that work on **any** probed
quantity:

- `grid_refine(build, n, probe)` solves at $N$ and $2N$ and reports the relative change,
- `auto_refine(build, n_start, probe, tol=...)` keeps doubling until every probed quantity
  settles below `tol`, or reports `converged = False` if it hits the cap.

Both take a `build(N)` callable and a `probe(solved)` returning a dict of scalars, so they are
element-agnostic.

The interesting question here is whether the *self-convergence* estimate they report is a
trustworthy proxy for the *true* error we can measure in this case.  Below we converge the horn
transmission $|T_{11}|$ at 400 Hz to 1 %, then check what the true error against the Webster
reference actually turned out to be.

In [ ]:
F_PROBE = 400.0


def probe_t11(n):
    "Magnitude of the (1,1) transfer-matrix entry at the probe frequency."
    t_n, _, _ = nefes_tm(n, np.array([F_PROBE]))
    return {"absT11": float(np.abs(t_n[0, 0, 0]))}


# one doubling, reported explicitly
gr = grid_refine(probe_t11, 16, lambda d: d)
print(
    f"grid_refine  N = {gr.n_coarse} -> {gr.n_fine}:  |T11| {gr.coarse['absT11']:.5f} -> "
    f"{gr.fine['absT11']:.5f}   rel change = {gr.worst:.2e}\n"
)

# refine until it settles
ar = auto_refine(probe_t11, 4, lambda d: d, tol=1e-2, max_refine=8)
print(
    f"auto_refine  converged = {ar.converged}  at N = {ar.n_final} "
    f"after {ar.n_refine} doublings   |T11| = {ar.final['absT11']:.5f}"
)
print(f"   self-convergence estimate at the last doubling : {ar.worst:.2e}")

t_ref = webster_tm(F_PROBE, RHO0, C0)
true_err = abs(ar.final["absT11"] - abs(t_ref[0, 0])) / abs(t_ref[0, 0])
print(f"   true error against the Webster reference       : {true_err:.2e}")
print(f"\nWebster |T11| at {F_PROBE:.0f} Hz = {abs(t_ref[0, 0]):.5f}")

The self-convergence estimate and the true error are the same order of magnitude, which is the
useful property: the refinement helpers report roughly what you are actually getting wrong.

For a first-order scheme that is expected — with error $\propto 1/N$, the change across a
doubling is itself $\propto 1/N$, so it estimates the remaining error up to a constant near one.
Do not read this as a guarantee: `acoustic_refinement.ipynb` shows a quantity with a shallow
plateau at coarse $N$, where a loose tolerance can stop early on the plateau rather than on the
answer. Converge the quantity, frequency, and tolerance you actually care about.

## What this establishes

- The taper composite converges on the **true horn**, not merely on itself: the interface-matching
  area changes are not just structurally present, they are numerically right.
- The rate is **first order** in $1/N$, matching the documented behaviour of the endpoint-biased
  staircase.
- The built-in refinement helpers give an error estimate of the right order, so $N$ can be chosen
  on a case where no reference solution exists.

The same comparison runs as a regression test in `tests/test_webster_horn.py`.